# Module 14 — Reinforcement Learning for Protein Design

## TL;DR — Plain English

**What is Reinforcement Learning?**
RL is a framework where an **agent** learns to make decisions by interacting with an **environment**, receiving **rewards** for good actions. Think of a chess AI: it doesn't memorize every possible game — it learns which moves tend to lead to winning positions through millions of self-play games.

**Why does protein design need RL?**
The protein sequence space is astronomically large: 20^100 possible sequences for a 100-residue protein. We cannot enumerate or brute-force this space. RL gives us a principled way to **navigate sequence space intelligently** — just as a chess AI navigates board states. The agent explores mutations, receives rewards based on predicted stability/binding/solubility, and learns which mutations are worth making.

**The analogy:**
- Chess AI: state = board position, action = move a piece, reward = win/lose
- Protein designer: state = current sequence, action = mutate position i to amino acid a, reward = ΔΔG / pLDDT / docking score

**Key algorithms covered:**
| Algorithm | Key Idea | Use Case |
|-----------|----------|----------|
| Q-Learning / DQN | Learn Q(s,a) value table / neural net | Discrete sequence mutations |
| REINFORCE | Policy gradient — optimize policy directly | End-to-end sequence generation |
| PPO | Clipped surrogate — stable, sample-efficient | Production protein design loops |
| Actor-Critic | Policy + value function combined | Reduced variance baseline |
| GFlowNets | Sample proportional to reward (diversity!) | Drug discovery, diverse candidates |

**After this notebook you will be able to:**
1. Formulate protein design as a Markov Decision Process (MDP)
2. Implement DQN, REINFORCE, and PPO from scratch
3. Explain GFlowNets and why they outperform standard RL for drug discovery
4. Handle multi-objective optimization with Pareto fronts
5. Confidently answer RL interview questions at computational biology ML teams / structural biology research labs

## Beginner Teaching Frame

**Who should fully work through this notebook:** advanced students. Beginners should use this as an elective concept notebook.

**How to study it on a first pass:**
- frame the biological task as state, action, reward
- learn the difference between value-based and policy-gradient methods
- treat PPO and GFlowNets as conceptual tools before implementation targets

**Common traps:** getting lost in RL notation before you can explain the protein-design task itself.


## Canonical Resource Upgrade

Best companion resources:
- [Sutton and Barto RL book](http://incompleteideas.net/book/the-book-2nd.html)
- [David Silver RL course](https://www.youtube.com/playlist?list=PLqYmG7hTraZBKeNJ-JE_eyJHZ7XgBoAyb)
- [OpenAI Spinning Up](https://spinningup.openai.com/en/latest/)


## 📄 Primary Literature — Key Papers

These results are based on peer-reviewed publications. Read the originals to go deeper:

- **Bengio et al. 2023** — *GFlowNet Foundations*  
  [https://arxiv.org/abs/2111.09266](https://arxiv.org/abs/2111.09266)
- **Schulman et al. 2017** — *Proximal Policy Optimization Algorithms*  
  [https://arxiv.org/abs/1707.06347](https://arxiv.org/abs/1707.06347)


## Cross-References

### Prerequisites (review these first)
- **`05_deep_learning_finetuning/01_dl_and_finetuning.ipynb`** — Neural network policies (the "brain" of the RL agent), backpropagation, PyTorch `nn.Module`
- **`12_diffusion_models/01_diffusion_protein_design.ipynb`** — Generative exploration of sequence space; diffusion and RL are complementary approaches to the same problem
- **`06_structural_ml_gnns/01_gnns_protein_graphs.ipynb`** — Protein graph reward signals; GNN-based oracles that score candidate sequences
- **`10_openfold3_finetuning/01_finetuning_capstone.ipynb`** — Sequence optimization objectives (ΔΔG, pLDDT, interface score) that become RL reward functions

### Next Steps
- **`10_openfold3_finetuning/01_finetuning_capstone.ipynb`** — TCR-pMHC optimization where RL loops over ProteinMPNN + AlphaFold3 oracle
- **`15_bayesian_optimization/01_bo_protein_design.ipynb`** — Bayesian optimization as an alternative to RL when oracle calls are expensive

### Key papers for this module
- DQN: Mnih et al. (2015) *Nature* — Playing Atari from pixels
- PPO: Schulman et al. (2017) arXiv:1707.06347
- GFlowNets: Bengio et al. (2021) arXiv:2106.04399
- REINVENT: Olivecrona et al. (2017) — RL for de novo drug design

## Section 1: RL Fundamentals — The MDP Framework

In [ ]:
import numpy as np

# RL fundamentals: MDP framework
print("Reinforcement Learning: The MDP Framework")
print("=" * 60)
print()

# State space for protein design
class ProteinDesignMDP:
    """MDP for protein sequence design.
    State: current partial sequence
    Action: choose next amino acid (20 choices)
    Reward: only at termination — oracle score (ESM log-likelihood + foldability)
    Episode: add one AA at a time until full sequence
    """
    def __init__(self, target_length=20):
        self.L = target_length
        self.aa = list('ACDEFGHIKLMNPQRSTVWY')
        self.reset()

    def reset(self):
        self.seq = []
        return tuple(self.seq)

    def step(self, action):
        self.seq.append(self.aa[action])
        done = len(self.seq) == self.L
        reward = self._compute_reward() if done else 0.0
        return tuple(self.seq), reward, done

    def _compute_reward(self):
        # Simulate: reward = ESM log-likelihood + GC content proxy
        seq = ''.join(self.seq)
        hydrophobic = sum(aa in 'VILMFYW' for aa in seq) / len(seq)
        # Good protein: ~40-60% hydrophobic
        return -abs(hydrophobic - 0.5) * 2

# Example episode
np.random.seed(42)
mdp = ProteinDesignMDP(target_length=10)
state = mdp.reset()
print("Episode trace:")
total_reward = 0
for step in range(10):
    action = np.random.randint(0, 20)
    state, reward, done = mdp.step(action)
    print(f"  Step {step+1}: action={mdp.aa[action]}, seq={''.join(state):10s}, reward={reward:.3f}")
    total_reward += reward
    if done: break
print(f"Total reward: {total_reward:.3f}")

## Section 2: Q-Learning and Deep Q-Networks (DQN)

### 📖 Reading Guide — Q-Learning Update Rule

`Q[state, action] = old_value + lr * (reward + gamma * max_Q_next - old_value)`
→ *The Bellman equation: update Q towards 'what I just got + discounted best future value'. lr=learning rate, gamma=discount.*

`gamma = 0.99`
→ *Discount factor: future rewards are worth 99% of immediate rewards. gamma=1 means value all future rewards equally; gamma=0 means only care about immediate reward.*

`epsilon = max(epsilon_min, epsilon * epsilon_decay)`
→ *Epsilon-greedy exploration: with probability epsilon, take a random action. Gradually reduce epsilon so the agent exploits its learned policy more over time.*

`target = reward + gamma * torch.max(target_net(next_state))`
→ *Target Q-value: what the Q-network should output for this state-action pair.*

`loss = F.mse_loss(current_Q, target.detach())`
→ *.detach() stops gradients flowing into the target network — it's a fixed reference, not trained in this step.*



In [ ]:
import torch
import torch.nn as nn
import numpy as np
from collections import deque
import random

# DQN for discrete protein design
class DQN(nn.Module):
    """Deep Q-Network for protein sequence design."""
    def __init__(self, state_dim=20*10, n_actions=20, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )

    def forward(self, state):
        return self.net(state)

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, done = zip(*batch)
        return (torch.stack(s), torch.tensor(a),
                torch.tensor(r, dtype=torch.float),
                torch.stack(s_next),
                torch.tensor(done, dtype=torch.float))

    def __len__(self):
        return len(self.buffer)

# Training setup
torch.manual_seed(42)
L, n_aa = 10, 20
state_dim = L * n_aa  # one-hot encoding

q_net = DQN(state_dim=state_dim, n_actions=n_aa)
target_net = DQN(state_dim=state_dim, n_actions=n_aa)
target_net.load_state_dict(q_net.state_dict())

optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)
buffer = ReplayBuffer(1000)

# Simulate a few training steps
def encode_state(seq, L=10, n_aa=20):
    x = torch.zeros(L * n_aa)
    for i, aa_idx in enumerate(seq):
        x[i*n_aa + aa_idx] = 1.0
    return x

print("DQN for protein design:")
print(f"  State: one-hot sequence ({state_dim} dims)")
print(f"  Actions: {n_aa} amino acids")
print(f"  Q-network: {sum(p.numel() for p in q_net.parameters()):,} params")
print(f"  Target network updated every 100 steps (stability)")
print(f"  ε-greedy exploration: ε decays 1.0 → 0.01 over training")

## Section 3: REINFORCE / Policy Gradients

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# REINFORCE / Policy Gradient
class PolicyNetwork(nn.Module):
    """Policy π_θ(a|s) for protein sequence design."""
    def __init__(self, state_dim=400, n_actions=20, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )

    def forward(self, state):
        logits = self.net(state)
        return torch.softmax(logits, dim=-1)

    def get_action(self, state):
        probs = self.forward(state)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob

def reinforce_update(policy, optimizer, episodes, gamma=0.99):
    """REINFORCE: update policy using MC returns."""
    all_log_probs = []
    all_returns = []

    for log_probs, rewards in episodes:
        # Compute discounted returns
        G = 0
        returns = []
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # normalize
        all_log_probs.extend(log_probs)
        all_returns.extend(returns)

    # REINFORCE loss: -E[G * log π(a|s)]
    loss = -torch.stack(all_log_probs).dot(torch.stack(all_returns))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()

torch.manual_seed(42)
policy = PolicyNetwork(state_dim=400, n_actions=20)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)

# Simulate episodes
episodes = []
for ep in range(10):
    log_probs, rewards = [], []
    state = torch.zeros(400)
    for step in range(5):  # short episode for demo
        a, lp = policy.get_action(state)
        reward = np.random.randn() * 0.1
        log_probs.append(lp); rewards.append(reward)
    episodes.append((log_probs, rewards))

loss = reinforce_update(policy, optimizer, episodes)
print(f"REINFORCE loss: {loss:.4f}")
print(f"Policy parameters: {sum(p.numel() for p in policy.parameters()):,}")
print("REINFORCE: high variance → use baselines, actor-critic, PPO")

## Section 4: PPO (Proximal Policy Optimization) for Protein Design

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# PPO for protein design
class ActorCritic(nn.Module):
    """Shared network for PPO actor (policy) and critic (value function)."""
    def __init__(self, state_dim=400, n_actions=20, hidden=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU()
        )
        self.actor  = nn.Linear(hidden, n_actions)  # policy head
        self.critic = nn.Linear(hidden, 1)           # value head

    def forward(self, state):
        h = self.shared(state)
        return torch.softmax(self.actor(h), dim=-1), self.critic(h).squeeze(-1)

def ppo_loss(old_log_probs, log_probs, advantages, values, returns,
             clip_eps=0.2, vf_coef=0.5, ent_coef=0.01):
    """PPO clipped objective."""
    ratio = torch.exp(log_probs - old_log_probs)
    # Clipped surrogate objective
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()
    # Value function loss
    value_loss = nn.MSELoss()(values, returns)
    # Entropy bonus (exploration)
    entropy = -(torch.exp(log_probs) * log_probs).mean()
    return policy_loss + vf_coef*value_loss - ent_coef*entropy

torch.manual_seed(42)
model = ActorCritic(state_dim=100, n_actions=20)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# Simulate PPO update
B = 32
state = torch.randn(B, 100)
probs, values = model(state)
dist = torch.distributions.Categorical(probs)
actions = dist.sample()
log_probs = dist.log_prob(actions)
advantages = torch.randn(B)  # simulated
returns = values.detach() + advantages

loss = ppo_loss(log_probs.detach(), log_probs, advantages, values, returns)
optimizer.zero_grad(); loss.backward(); optimizer.step()
print(f"PPO loss: {loss.item():.4f}")
print(f"PPO used in: RFdiffusion sampling, AlphaFold3 finetuning, ProteinMPNN-RL")
print(f"Key: clip_eps prevents large policy updates → more stable than REINFORCE")

## Section 5: GFlowNets — RL-Inspired Generative Models

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# GFlowNets for diverse molecule generation
print("GFlowNets: Generative Flow Networks")
print("=" * 60)
print()
print("Key idea: GFlowNet ≈ RL where we want π(x) ∝ R(x)^(1/T)")
print("NOT the argmax — sample in proportion to reward!")
print()
print("Applications in protein design:")
print("  * Diverse antibody CDR generation")
print("  * Multi-objective drug candidate sampling")
print("  * Exploration in protein sequence space")
print()
print("Trajectory balance (TB) objective:")
print("  Z * π_F(τ) = R(x) * π_B(τ)  for complete trajectories τ")
print("  Equivalent to: log Z + Σ log π_F(s_t→s_{t+1}) = log R(x) + Σ log π_B")
print()

class GFlowNetTB(nn.Module):
    """GFlowNet with Trajectory Balance objective.
    Simplified: binary strings of length L, reward = hamming weight.
    """
    def __init__(self, L=5, hidden=64):
        super().__init__()
        self.L = L
        # Forward policy: at each step, choose 0 or 1
        self.forward_policy = nn.Sequential(
            nn.Linear(L+1, hidden), nn.ReLU(),  # state + step
            nn.Linear(hidden, 2)
        )
        # Backward policy
        self.backward_policy = nn.Sequential(
            nn.Linear(L+1, hidden), nn.ReLU(),
            nn.Linear(hidden, 2)
        )
        self.log_Z = nn.Parameter(torch.tensor(0.0))

    def reward(self, seq):
        """R(x) = exp(hamming_weight / temperature)"""
        return torch.exp(seq.float().sum(-1) / 2)

torch.manual_seed(42)
model = GFlowNetTB(L=6)
params = sum(p.numel() for p in model.parameters())
print(f"GFlowNet params: {params:,}")
print(f"log_Z (partition function): {model.log_Z.item():.3f}")
print()
print("GFlowNets vs RL policy gradients:")
print("  RL: maximize expected reward (finds single mode)")
print("  GFlowNet: sample diverse high-reward solutions")
print("  → Better for drug discovery where diversity matters")

## Section 6: Multi-Objective RL and Pareto Optimization

In [ ]:
import numpy as np
import torch

# Multi-objective RL and Pareto optimization
print("Multi-Objective RL: Pareto Optimization for Protein Design")
print("=" * 60)
print()
print("Protein design objectives (often conflicting):")
objectives = {
    "Stability (ΔG)": "Lower = more stable (thermostable)",
    "Solubility": "Higher = less aggregation",
    "Binding affinity (Kd)": "Lower = tighter binding",
    "Immunogenicity": "Lower = less immune response",
    "Expression level": "Higher = more protein produced",
}
for obj, desc in objectives.items():
    print(f"  {obj}: {desc}")

print()
# Pareto front computation
def is_pareto_dominated(costs):
    """Return boolean mask of non-dominated (Pareto-optimal) points."""
    n = len(costs)
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i != j:
                # j dominates i if j is better in ALL objectives
                if np.all(costs[j] <= costs[i]) and np.any(costs[j] < costs[i]):
                    is_pareto[i] = False
                    break
    return is_pareto

np.random.seed(42)
n_candidates = 100
# Two objectives: stability (lower better) and binding affinity (lower better)
costs = np.random.rand(n_candidates, 2)
pareto_mask = is_pareto_dominated(costs)

pareto_front = costs[pareto_mask]
print(f"Generated {n_candidates} protein candidates")
print(f"Pareto-optimal candidates: {pareto_mask.sum()}")
print(f"Non-dominated front size: {len(pareto_front)}")
print()
print("MORL approaches:")
approaches = [
    "Scalarization: w1*obj1 + w2*obj2 (simple but misses non-convex front)",
    "NSGA-II / NSGA-III: evolutionary multi-objective optimization",
    "MORL with Pareto-hypervolume reward",
    "ContextualBO: Bayesian optimization with multi-objective acquisition",
]
for a in approaches:
    print(f"  • {a}")

## Section 7: Interview Q&A + Time Complexity

### Interview Q&A

**Q: What is the difference between model-based and model-free RL?**

A: Model-free (DQN, PPO) learns directly from experience. Model-based (AlphaZero, Dreamer) learns a transition model P(s'|s,a) first, enabling planning. For protein design, oracle calls (wet lab) are expensive — model-based RL is preferred when a good simulator exists (MD, Rosetta).

---

**Q: Why does REINFORCE have high variance?**

A: The gradient estimator `grad J = E[grad log pi(a|s) * G_t]` averages over complete trajectories. `G_t` includes future rewards which are noisy. Solution: subtract a baseline b(s) that does not depend on action: `G_t - b(s)`. The baseline has zero expectation effect but reduces variance dramatically.

---

**Q: PPO vs TRPO — what is the practical difference?**

A: TRPO (Trust Region Policy Optimization) uses a hard KL constraint solved via conjugate gradient — mathematically rigorous but complex. PPO approximates it with a simple clip operation, making PPO easier to implement and equally effective in practice. PPO is the industry default.

---

**Q: What is the credit assignment problem in protein design?**

A: A single mutation's effect depends on 20+ other positions (epistasis). A sequence of 50 mutations may only be rewarded after wet-lab validation — a delayed reward spanning weeks. Solutions: (1) learned reward models (proxy functions like ESM2 embeddings + regressor), (2) intermediate rewards from structure predictors (pLDDT as a proxy for stability).

---

**Q: Why GFlowNets over standard RL for drug discovery?**

A: Standard RL maximizes expected reward, leading to mode collapse — the agent converges to a single best candidate. GFlowNets sample proportional to reward, generating a diverse library of high-reward candidates. Diversity is critical since many candidates fail in clinical trials for off-target reasons unrelated to the reward signal.

---

**Q: What is the Bellman equation and why is it central to Q-learning?**

A: `Q(s,a) = r + gamma * max_{a'} Q(s',a')`. It expresses that the value of a state-action pair equals the immediate reward plus the discounted value of the best next action. Q-learning uses this as a bootstrapped target — we learn Q by iteratively making it consistent with itself (temporal difference learning).

---

### Time Complexity

| Algorithm | Per-step Cost | Memory | Sample Efficiency | Notes |
|-----------|--------------|--------|-------------------|-------|
| Q-learning (tabular) | O(1) | O(\|S\|\|A\|) | Low | Only for tiny state spaces |
| DQN | O(d^2) forward pass | O(buffer) | Medium | Experience replay critical |
| REINFORCE | O(T * d^2) | O(T) | Low | High variance; baseline helps |
| PPO | O(T * d^2 * epochs) | O(T) | High | Industry standard |
| AlphaZero | O(MCTS * d^2) | O(tree) | Very High | Requires fast simulator |
| GFlowNets | O(T * d^2) | O(T) | High | Best for diverse generation |

Where d = hidden dimension, T = trajectory length.

## Resources

### 1. Theory
- Sutton & Barto "Reinforcement Learning: An Introduction" (2nd ed, free PDF): http://incompleteideas.net/book/the-book-2nd.html
- David Silver RL lectures (UCL/structural biology research labs, 10 lectures): https://www.davidsilver.uk/teaching/
- Lilian Weng RL blog series: https://lilianweng.github.io/posts/2018-02-19-rl-overview/
- Spinning Up in Deep RL (OpenAI): https://spinningup.openai.com/
- PPO paper (Schulman 2017): https://arxiv.org/abs/1707.06347
- GFlowNets tutorial (Bengio 2021): https://arxiv.org/abs/2106.04399
- DQN paper (Mnih 2015, Nature): https://www.nature.com/articles/nature14236

### 2. Must-Have Popular Resources

**Start Here (Zero Background):**
- 3Blue1Brown "But what is a Neural Network?" — watch before starting RL
- David Silver Lecture 1 (30 min intro, no prerequisites): https://www.youtube.com/watch?v=2pWv7GOvuf0
- OpenAI Spinning Up documentation (conceptual overview): https://spinningup.openai.com/en/latest/
- StatQuest: "Reinforcement Learning Explained" YouTube: https://www.youtube.com/c/joshstarmer

**Popular / Advanced:**
- "Deep RL Doesn't Work Yet" (honest perspective): https://www.alexirpan.com/2018/02/14/rl-hard.html
- Hugging Face Deep RL Course (free, certificates): https://huggingface.co/learn/deep-rl-course/
- CleanRL (clean single-file RL implementations): https://github.com/vwxyzjn/cleanrl
- Stable Baselines3 (production RL library): https://github.com/DLR-RM/stable-baselines3

### 3. Practicals
- HuggingFace Deep RL Course with hands-on labs: https://huggingface.co/learn/deep-rl-course/
- RL4CO (RL for combinatorial optimization, protein-adjacent): https://github.com/ai4co/rl4co
- REINVENT4 (AstraZeneca RL for drug design): https://github.com/MolecularAI/REINVENT4
- Gymnasium (OpenAI Gym successor): https://github.com/Farama-Foundation/Gymnasium
- Kaggle: "RL for trading" as proxy for sequence optimization patterns
- MolRL-MGPT (molecular RL with GPT): https://github.com/futianfan/reinforced-genetic-algorithm

### 4. Real-World Problems

1. **De novo protein design with RL**: ProteinMPNN + PPO feedback loop for binding optimization. Dataset: ProteinGym substitution DMS data. [ProteinGym GitHub](https://github.com/OATML-Markslab/ProteinGym)

2. **Antibody CDR optimization**: REINVENT-style RL loop — ESM2 embedding as state, reward = structural model pLDDT + docking score. Dataset: [SAbDab](https://opig.stats.ox.ac.uk/webapps/newsabdab/sabdab/)

3. **RNA aptamer design**: Action = replace nucleotide, reward = predicted binding (RNAfold free energy). Dataset: Eterna RNA design puzzle dataset at https://eternagame.org/

4. **Small molecule optimization**: REINVENT4 tutorial on AstraZeneca GitHub — directly applicable patterns to peptide design.

### 5. Interview Tips
- Know the Bellman equation cold: `Q(s,a) = r + gamma * max_{a'} Q(s',a')`
- Explain the exploration-exploitation tradeoff: epsilon-greedy, UCB, Thompson sampling
- DQN tricks: experience replay (breaks correlation), target network (stabilizes targets)
- PPO: why clipping works better than KL penalty in practice (simpler, no second-order optimization)
- For protein design roles: always mention REINVENT4, GFlowNets, oracle budget, and multi-objective optimization
- Key papers to know: DQN (Mnih 2015), PPO (Schulman 2017), GFlowNets (Bengio 2021), REINVENT (Olivecrona 2017)
- Know GFlowNets: "sampling proportional to reward" — this differentiates you from generic RL candidates

### 6. Hot Industry Topics
- **GFlowNets for drug discovery**: Diverse candidate generation, used at Mila/drug discovery companies
- **RLHF (RL from Human Feedback)**: Foundation of ChatGPT; protein design analog = RLFP (RL from protein feedback, e.g., wet-lab oracle)
- **AlphaFold + RL**: Using AF3 as oracle in RL loop for structure-guided design — active computational biology ML teams research direction
- **Offline RL for biological data**: Learning from existing sequence-function datasets (ProteinGym, DMS) without active exploration — critical when wet lab is expensive
- **Constitutional AI / Safe RL**: Constraints on generated sequences (no predicted toxicity, no aggregation signals, no PAN-assay activity)
- **Multi-objective drug design**: MORL for simultaneous ADMET property optimization — industry standard at AstraZeneca, Novo Nordisk

## Timed Practice Problems

Work through these without looking at the notebook code above. Set a timer.

---

### Problem 1 — Epsilon-Greedy Action Selection (5 min)

Implement a function `epsilon_greedy(q_values, epsilon)` that:
- Takes a 1D array of Q-values and an epsilon parameter
- Returns the greedy action with probability `(1 - epsilon)`
- Returns a random action with probability `epsilon`

**Hint:** Use `np.random.random()` to sample a uniform [0,1] value, then branch.

```python
def epsilon_greedy(q_values, epsilon):
    # YOUR CODE HERE
    pass

# Test
q = np.array([1.0, 5.0, 2.0, 0.5])  # best action is index 1
actions = [epsilon_greedy(q, epsilon=0.1) for _ in range(1000)]
print(f"Greedy action (idx 1) chosen: {actions.count(1)/10:.1f}% of the time (expect ~90%)")
```

---

### Problem 2 — Discounted Return Calculation (3 min)

Given a list of rewards `[r0, r1, r2, ..., rT]` and discount factor `gamma`, compute the discounted return at each timestep: `G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...`

**Hint:** Iterate in reverse — `G_t = r_t + gamma * G_{t+1}`

```python
def compute_returns(rewards, gamma=0.99):
    # YOUR CODE HERE
    pass

# Test
rewards = [1.0, 0.0, 0.0, 10.0]  # sparse reward at end
G = compute_returns(rewards, gamma=0.9)
print(f"Returns: {[f'{g:.3f}' for g in G]}")  # expect [8.29, 8.1, 9.0, 10.0] approx
```

---

### Problem 3 — PPO Clipped Objective (2 min)

Write the PPO clipped surrogate objective as a single mathematical expression.

**Hint:** It involves a ratio `r_t(theta)`, an advantage `A_t`, and a clip operation. The final objective is a `min` of two terms.

**Answer (fill in the blanks):**

```
L_CLIP(theta) = E_t [ min( r_t(theta) * A_t,  clip(r_t(theta), _____, _____) * A_t ) ]

where r_t(theta) = pi_theta(a_t|s_t) / pi_{theta_old}(a_t|s_t)
```

---

### Problem 4 — Antibody VH Loop Reward Function (10 min — open-ended)

Design a reward function for optimizing an antibody VH CDR3 loop for:
- High binding affinity to a target antigen
- Low predicted immunogenicity (human-like sequence)
- Acceptable solubility

**Consider:**
1. What tools would you use for each component? (ESMFold, AbAgIntPre, SCRATCH-1D?)
2. How would you combine the three objectives? (weighted sum, constraint, Pareto?)
3. What intermediate rewards could you use to handle the sparse reward problem?
4. How would you handle the case where the structure predictor is slow (10 sec/call)?

**Write pseudocode** for a `compute_reward(sequence)` function that handles all these concerns.

---

### Problem 5 — Variance Reduction by Baseline (8 min)

Explain mathematically why subtracting a baseline `b(s)` from the return `G_t` reduces variance **without changing the expected gradient**.

**Hint:** The policy gradient theorem gives `grad J = E[grad log pi(a|s) * G_t]`. Show that `E[grad log pi(a|s) * b(s)] = 0` for any baseline that does not depend on the action `a`.

**Key identity to use:**
```
sum_a pi(a|s) * grad log pi(a|s) = grad sum_a pi(a|s) = grad 1 = 0
```

Write out the 3-4 line proof. This is a common computational biology ML teams / structural biology research labs interview question.

---

**Solutions:** Run the code cells above and compare your implementations. For Problems 4-5, discuss with a study partner or ask Claude: `"Review my reward function design for Problem 4 in 14/01"`

## Mastery Check

On a first pass, success means you can:
1. define the state, action, and reward in a protein-design framing
2. explain DQN vs PPO at a high level
3. explain why reward design is dangerous
4. explain why GFlowNets are interesting for diverse design generation


---
## ⚠️ Reward Hacking in Protein Design

The most important practical failure mode of RL for protein design: the model learns to maximize the proxy reward rather than actual fitness.


In [ ]:
# REWARD HACKING DEMONSTRATION
# This is the most important lesson in applying RL to protein design

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Scenario: Design a peptide to maximize "hydrophobicity score" as a proxy for binding
# True goal: maximize binding affinity
# Proxy: hydrophobicity score (fast to compute, correlated with binding)

def hydrophobicity_score(sequence):
    """Fast proxy reward: sum of hydrophobicity values (Kyte-Doolittle scale)."""
    kd = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,
          'G':-0.4,'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,
          'P':-1.6,'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
    return sum(kd.get(aa, 0) for aa in sequence) / len(sequence)

def true_binding_affinity(sequence):
    """True reward: actual binding (expensive, measured in the lab).

    For our simulation: weakly correlated with hydrophobicity,
    but requires BALANCE (not just high hydrophobicity).
    Highly hydrophobic peptides are insoluble and aggregate!
    """
    hydrpb = hydrophobicity_score(sequence)
    # True binding peaks at moderate hydrophobicity, drops for extreme values
    # (very hydrophobic peptides aggregate and can't reach target)
    optimal_hydrophobicity = 1.0  # optimal range
    binding = 5.0 * np.exp(-0.5 * (hydrpb - optimal_hydrophobicity)**2 / 0.8**2)
    # Add sequence complexity (diverse sequences bind better)
    n_unique = len(set(sequence)) / len(sequence)
    binding += 2.0 * n_unique
    return binding + np.random.normal(0, 0.1)

# Simulate RL optimization: generation 0 = random, successive = optimized for proxy
amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
seq_len = 20

def random_sequence():
    return ''.join(np.random.choice(amino_acids, seq_len))

def optimize_by_proxy(initial_seq, n_rounds=50):
    """Simple hill-climbing on proxy reward (simulates RL behavior)."""
    seq = list(initial_seq)
    proxy_history = [hydrophobicity_score(seq)]
    true_history  = [true_binding_affinity(seq)]

    for _ in range(n_rounds):
        # Mutate: change a random position
        pos = np.random.randint(seq_len)
        new_aa = np.random.choice(amino_acids)
        old_aa = seq[pos]
        seq[pos] = new_aa

        # Accept if proxy reward improves
        new_proxy = hydrophobicity_score(seq)
        if new_proxy > proxy_history[-1]:
            proxy_history.append(new_proxy)
            true_history.append(true_binding_affinity(seq))
        else:
            seq[pos] = old_aa  # Reject
            proxy_history.append(proxy_history[-1])
            true_history.append(true_history[-1])

    return proxy_history, true_history, ''.join(seq)

# Run multiple trials
n_trials = 20
proxy_final, true_final, seqs_final = [], [], []

for _ in range(n_trials):
    init_seq = random_sequence()
    p_hist, t_hist, final_seq = optimize_by_proxy(init_seq, n_rounds=200)
    proxy_final.append(p_hist[-1])
    true_final.append(t_hist[-1])
    seqs_final.append(final_seq)

# Compare initial vs optimized
init_seqs = [random_sequence() for _ in range(100)]
init_proxy = [hydrophobicity_score(s) for s in init_seqs]
init_true  = [true_binding_affinity(s) for s in init_seqs]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Proxy vs True binding scatter
axes[0].scatter(init_proxy, init_true, alpha=0.4, s=20, color='blue', label='Random seqs')
axes[0].scatter(proxy_final, true_final, alpha=0.7, s=60, color='red', marker='*',
               label='After proxy optimization')
axes[0].set_xlabel('Proxy Reward (Hydrophobicity)')
axes[0].set_ylabel('True Binding Affinity')
axes[0].set_title('Reward Hacking: Proxy vs True Reward')
axes[0].legend()

# 2. Distribution of true binding
axes[1].hist(init_true, bins=15, alpha=0.6, color='blue', label='Random')
axes[1].hist(true_final, bins=15, alpha=0.6, color='red', label='Proxy-optimized')
axes[1].set_xlabel('True Binding Affinity')
axes[1].set_ylabel('Count')
axes[1].set_title('Does Proxy Optimization Improve True Binding?')
axes[1].legend()

# 3. Amino acid composition of "optimized" sequences
aa_counts = {}
for seq in seqs_final:
    for aa in seq:
        aa_counts[aa] = aa_counts.get(aa, 0) + 1
sorted_aa = sorted(aa_counts.items(), key=lambda x: -x[1])
axes[2].bar([aa for aa, _ in sorted_aa[:10]],
           [cnt/sum(aa_counts.values()) for aa, cnt in sorted_aa[:10]],
           color='coral')
axes[2].axhline(1/20, color='gray', linestyle='--', label='Expected (uniform)')
axes[2].set_xlabel('Amino acid')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Reward Hacking: RL finds\nmostly hydrophobic AAs (I,L,V,F)')
axes[2].legend()

plt.tight_layout()
plt.savefig('reward_hacking.png', dpi=100)
plt.show()

print("REWARD HACKING ANALYSIS:")
init_mean_true = np.mean(init_true)
opt_mean_true  = np.mean(true_final)
print(f"  Random sequences: mean binding = {init_mean_true:.3f}")
print(f"  Proxy-optimized:  mean binding = {opt_mean_true:.3f}")
if opt_mean_true > init_mean_true:
    print(f"  Proxy worked! (+{opt_mean_true-init_mean_true:.3f})")
else:
    print(f"  REWARD HACKING CONFIRMED: proxy optimization hurt true binding!")

print()
print("FIX STRATEGIES:")
print("  1. Composite reward: 0.6*binding_proxy + 0.2*solubility + 0.2*diversity")
print("  2. Periodic wet-lab validation and reward model retraining (RLHF-style)")
print("  3. Constrained optimization: maximize proxy subject to solubility >= threshold")
print("  4. Use ESM-2 log-likelihood as proxy (better correlated with folded state)")
print()
print("KEY PAPER: Trabuco et al. (2022) 'Reward hacking in protein design with RL'")

---
## 🔗 Cross-Module Connections — How RL Fits Into the Protein ML Stack

### RL as the "Reward Signal" Layer

Reinforcement learning in protein design answers the question: **given a generative model, how do we steer it toward molecules with desired properties?**

| Module | RL Connection | How RL Is Used |
|--------|---------------|----------------|
| **12/01** Diffusion | RL fine-tunes diffusion models via RLHF | DPO/PPO used to steer RFdiffusion toward binders |
| **13/01** Bayesian | Thompson Sampling = Bayesian RL; GP-UCB = acquisition function | Bayesian optimization for protein sequence design |
| **15/01** SSL | Pre-trained ESM-2 is the "environment model" in latent-space RL | RL in latent space of protein LM = efficient search |
| **10/01** Fine-Tuning | RLHF (InstructGPT-style) applied to protein generation | Reward = ΔΔG or docking score; policy = protein LM |
| **07/05** AF3 | AF3 is the oracle/reward model in protein design RL loops | Predicted pLDDT / iPTM used as reward signal |

### The Protein Design Loop

```
START: Random or seed sequence
  ↓
GENERATOR (Module 12): Diffusion/flow matching → candidate structures
  ↓
ORACLE (Module 07): AF3 → pLDDT, PAE, iPTM confidence scores
  ↓
REWARD (Module 10): ΔΔG prediction or docking score
  ↓
POLICY UPDATE (Module 14): GFlowNets / PPO → update generator
  ↓
LOOP until convergence or compute budget exhausted
```

**Why GFlowNets over PPO for protein design:**
- PPO maximizes reward → mode-seeking → generates the same high-reward sequence repeatedly
- GFlowNets samples proportional to reward → mode-covering → diverse set of solutions
- For drug discovery, diversity matters: you want 100 different binder scaffolds, not 100 copies of the best one

### Connecting to Module 13

Bayesian optimization (Module 13) is the data-efficient version of this RL loop:
- Use a GP surrogate instead of running AF3 on every candidate (expensive oracle → query it rarely)
- Acquisition function (UCB/EI) = explicit exploration-exploitation tradeoff
- For <1000 experimental evaluations: Bayesian optimization beats RL
- For >10000 evaluations (virtual screening): RL/GFlowNets scale better than GPs


## 🐛 Debug Exercise — Reinforcement Learning for Protein Design

Find and fix the **3 bugs** in the RL training loop below.

**Expected correct output:**
```
Rewards after normalization: mean ~0, std ~1
Q-value update includes discount factor: Q = r + gamma * max(Q_next)
Epsilon-greedy: explores with probability epsilon (not 1 - epsilon)
```

<details>
<summary>Show answer</summary>

**Bug 1 — Reward not normalized:** Raw ΔΔG values can range from -20 to +20 kcal/mol.
Feeding these directly as rewards causes unstable training. Fix: normalize rewards to
mean=0, std=1 per batch using `(r - r.mean()) / (r.std() + 1e-8)`.

**Bug 2 — Q-update missing discount factor γ:** The Bellman equation is
`Q(s,a) = r + γ * max_a Q(s',a)`. Without γ, future rewards are not discounted,
causing instability. Fix: `target = reward + gamma * max_q_next`.

**Bug 3 — Epsilon-greedy inverted:** `if random.random() > epsilon` explores when the
random number is LARGE (> ε), meaning it explores with probability `1 - ε` — completely
backwards. Fix: `if random.random() < epsilon` to explore with probability ε.
</details>


In [ ]:
# DEBUG EXERCISE — RL protein design training loop, find and fix 3 bugs
import numpy as np
import random

np.random.seed(42)
random.seed(42)

# Simulated ΔΔG rewards (raw, large magnitude)
raw_rewards = np.array([-18.5, 2.3, -0.4, 15.7, -8.2, 4.1, -12.6, 7.9, 0.2, -5.5])

# BUG 1: rewards not normalized — large values destabilize training
rewards_for_training = raw_rewards   # should normalize: (r - r.mean()) / (r.std() + 1e-8)
print(f"Raw reward stats: mean={rewards_for_training.mean():.2f}, std={rewards_for_training.std():.2f}")
print("BUG: large magnitude rewards cause unstable gradient updates")
print(f"After normalization: mean should be ~0, std ~1")

# DQN Q-value update
Q_current = np.array([1.5, 2.3, 0.8, 3.1])   # Q-values for 4 actions in state s
Q_next    = np.array([2.0, 1.1, 3.5, 0.9])   # Q-values for next state s'
reward    = 0.5
gamma     = 0.99

# BUG 2: missing discount factor gamma in Bellman update
target_wrong   = reward + Q_next.max()          # BUG: missing gamma
target_correct = reward + gamma * Q_next.max()  # correct

print(f"\nQ-target (missing gamma): {target_wrong:.4f}")
print(f"Q-target (with gamma={gamma}): {target_correct:.4f}")
print(f"Difference: {target_correct - target_wrong:.4f} — significant over many steps!")

# Epsilon-greedy action selection
epsilon = 0.1  # 10% exploration probability
n_actions = 4
actions_taken = []
for _ in range(1000):
    # BUG 3: condition inverted — explores when random > epsilon (90% of the time!)
    if random.random() > epsilon:   # BUG: should be < epsilon
        action = random.randint(0, n_actions - 1)  # explore
    else:
        action = np.argmax(Q_current)              # exploit

    actions_taken.append(action)

explore_pct = sum(1 for a in actions_taken if a != np.argmax(Q_current)) / len(actions_taken)
print(f"\nExploration rate (epsilon={epsilon}, should be ~10%): {explore_pct*100:.1f}%")
print(f"BUG: inverted condition means exploring {explore_pct*100:.1f}% instead of {epsilon*100:.0f}%")
